# Lezione 02 — Testo grezzo e tokenizzazione

**Corso:** NLP Classico con Python · **Blocco 1:** Dal testo ai token
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/02_texto_crudo_tokenizacion_IT.md`

## Cosa vedrai in questa lezione

- **Unicode**: perché lo stesso carattere accentato può avere due rappresentazioni (NFC e NFD), come lo rivela `len()` e quando togliere gli accenti è un errore.
- **Pulizia con espressioni regolari** (`re`) senza distruggere informazione utile: URL, menzioni, allungamenti, emoji, cifre, residui di markup. Costruirai una funzione `limpiar(texto)` passo per passo.
- **Tokenizzazione**: perché `split()` non basta; `re.findall`, NLTK e spaCy a confronto; segmentazione in frasi.
- **Token e tipi**: dimensione del vocabolario, frequenze, hapax e la distribuzione rango-frequenza in due testi letterari di pubblico dominio e nelle recensioni.
- **Misurare la qualità di una pulizia** rispetto alla colonna `texto_limpio` (riferimento) delle recensioni.
- **Altre difficoltà del linguaggio** (anafora, pragmatica, variazione) e perché nessuna pulizia le risolve.

## Prerequisiti

- Python di base e la lezione 1 (la pipeline classica e il passo 3 «ingenuo» che qui si corregge).
- Librerie: `pandas`, `matplotlib`, `nltk` (con le risorse `punkt`) e `spaCy` (solo il suo tokenizzatore, senza modelli). La cella seguente le installa solo se mancano (funziona allo stesso modo in Google Colab).
- La cartella `datos/` del corso (in Colab, caricala nel pannello dei file).

> **Come usare questo notebook:** esegui le celle in ordine. Tutte le cifre provengono da questi corpora piccoli e di stile controllato (*esempio illustrativo*): non si generalizzano a testi reali di qualunque provenienza.

---
## Installazione e risorse

Questa cella installa ciò che manca e scarica le risorse di NLTK. Con tutto installato non fa nulla.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

for paquete, modulo in (("pandas", "pandas"), ("matplotlib", "matplotlib"), ("nltk", "nltk"), ("spacy", "spacy")):
    asegurar(paquete, modulo)

import nltk

for recurso in ("punkt", "punkt_tab"):
    try:
        nltk.data.find(f"tokenizers/{recurso}")
    except LookupError:
        nltk.download(recurso, quiet=True)

print("Dipendenze e risorse di NLTK pronte (di spaCy si usa solo il tokenizzatore, senza scaricare modelli).")

---
## Cella 0 — Setup

Importiamo le librerie, fissiamo il seme e definiamo `cargar_datos(nombre)`, che cerca il file in `datos/`, `../datos/` e `/content/datos/`. Carichiamo le 300 recensioni per lingua e i due estratti letterari di pubblico dominio (il capitolo I del *Don Chisciotte* e le prime ≈ 1 900 parole del capitolo I de *I promessi sposi*). Sono testi di secoli e stili diversi: servono a vedere il comportamento del testo «pulito» e di quello rumoroso.

In [ ]:
import os
import re
import unicodedata
import difflib
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize

random.seed(42)                          # seme fisso: tutti otteniamo gli stessi risultati
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

def cargar_datos(nombre):
    """Cerca un file in datos/, ../datos/ e /content/datos/ e ne restituisce il percorso."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(f"Non trovo '{nombre}'. Carica la cartella datos/ nell'ambiente (in Colab: pannello dei file) oppure lasciala accanto al corso.")

resenas = {"spa": pd.read_csv(cargar_datos("resenas_es.csv")), "ita": pd.read_csv(cargar_datos("resenas_it.csv"))}
libros = {}
for idioma, archivo in (("spa", "gutenberg/quijote_cap1.txt"), ("ita", "gutenberg/promessi_sposi_cap1.txt")):
    with open(cargar_datos(archivo), encoding="utf-8") as f:
        libros[idioma] = f.read()

print({i: len(d) for i, d in resenas.items()}, {i: len(t) for i, t in libros.items()}, "(recensioni e caratteri di ogni libro)")

---
## 1. Unicode: che cos'è davvero un «carattere»

### 1.1 Punti di codice e byte

Unicode assegna a ogni simbolo un **punto di codice** (`U+00F3` per `ó`). UTF-8 è il modo di salvarli in byte: le lettere ASCII occupano 1 byte e quelle accentate 2 (le emoji, 4). Per questo `len()` conta punti di codice e **non** byte né «ciò che vede l'occhio».

In [ ]:
# datos-es
PALABRAS_UNICODE = {"spa": "canción", "ita": "città"}

In [ ]:
for idioma, palabra in PALABRAS_UNICODE.items():
    print(idioma, palabra, "→", len(palabra), "punti di codice,", len(palabra.encode("utf-8")), "byte in UTF-8")

for c in PALABRAS_UNICODE["ita"]:
    print(f"{c!r:5} U+{ord(c):04X}  {unicodedata.name(c)}")

emoji = "\U0001F60A"
print(emoji, len(emoji), "punto di codice,", len(emoji.encode("utf-8")), "bytes")

### 1.2 Esperimento guidato: due modi di scrivere la stessa parola

`é` può essere salvata come **un** punto di codice (forma composta, **NFC**: `U+00E9`) oppure come **due**: la `e` e un accento combinante `U+0301` (forma scomposta, **NFD**). Le due si disegnano uguali. Confrontiamo le due versioni di `canción`.

In [ ]:
nfc = unicodedata.normalize("NFC", PALABRAS_UNICODE["spa"])
nfd = unicodedata.normalize("NFD", PALABRAS_UNICODE["spa"])

print(nfc, nfd, "→ sembrano uguali, ma len:", len(nfc), "vs", len(nfd))
print("Sono uguali per Python?", nfc == nfd)
print("Punti di codice della versione NFD:", [f"U+{ord(c):04X}" for c in nfd])

indice = {nfc: "chiave salvata in NFC"}
print("Trovo la versione NFD nel dizionario?", nfd in indice)
print("E se normalizzo prima di cercare?", unicodedata.normalize("NFC", nfd) in indice)

**Cosa osservi?** Le due stringhe sembrano identiche, ma hanno lunghezze diverse (7 e 8) e `==` dà `False`. Un dizionario, un `Counter` o un vocabolario tratterebbero `canción` come **due parole**. Succede nella pratica: i nomi dei file di macOS e alcuni testi copiati da PDF o dal web arrivano in NFD. **Regola:** normalizza tutto in NFC appena lo leggi.

Esiste anche **NFKC**, che applica pure equivalenze di «compatibilità» (legature, apici, frazioni). È utile, ma cambia più cose (`²` → `2`), quindi si usa con criterio.

In [ ]:
# datos-es
COMPATIBILIDAD = ["ﬁnestra", "m²", "½", "…"]

In [ ]:
for texto in COMPATIBILIDAD:
    print(f"{texto!r:12} NFC → {unicodedata.normalize('NFC', texto)!r:12} NFKC → {unicodedata.normalize('NFKC', texto)!r}")

### 1.3 `lower()` contro `casefold()`

Per **confrontare** testi senza distinguere le maiuscole, `casefold()` è più aggressivo di `lower()`: pensato per confrontare, non per mostrare. In spagnolo e italiano quasi sempre coincidono, ma ci sono casi reali (legature copiate da PDF, la `ß` tedesca nei nomi propri).

In [ ]:
# datos-es
CASOS_MAYUS = [("Straße", "STRASSE"), ("ﬁnestra", "FINESTRA"), ("Roma", "ROMA")]

In [ ]:
for a, b in CASOS_MAYUS:
    print(f"{a!r:10} {b!r:10}  lower ==: {a.lower() == b.lower()!s:5}  casefold ==: {a.casefold() == b.casefold()}")

**Cosa osservi?** Con `lower()` la legatura `ﬁ` e la `ß` non coincidono con i loro equivalenti; con `casefold()` sì. Ma attenzione all'altra faccia: passando tutto in minuscolo `Rosa` (nome) e `rosa` (colore) diventano la **stessa** stringa, e anche `Marino` (cognome) e `marino` (aggettivo). Passare al minuscolo è una decisione, non un automatismo.

### 1.4 Togliere gli accenti… e quando no?

Togliere gli accenti riduce le varianti (`rapido`/`rápido`), ma **cancella distinzioni**: `papá`/`papa`, `sí`/`si`, `él`/`el`, `è`/`e`, `dà`/`da`. E un'implementazione ingenua si porta via la `ñ`: `año` → `ano`. Scriviamo una versione che protegge la `ñ` e misuriamo quante parole **distinte** del vocabolario delle recensioni pulite si fondono in una sola togliendo gli accenti.

In [ ]:
def quitar_acentos(texto):
    """Toglie gli accenti (segni combinanti) e conserva la tilde della n."""
    salida = []
    for c in unicodedata.normalize("NFD", texto):
        if unicodedata.category(c) == "Mn" and not (c == "̃" and salida and salida[-1] in "nN"):
            continue
        salida.append(c)
    return unicodedata.normalize("NFC", "".join(salida))

def tipos_en_colision(textos):
    """Raggruppa le parole distinte che coincidono togliendo gli accenti."""
    frecuencia = Counter(p for t in textos for p in re.findall(r"\w+", t.lower()))
    grupos = {}
    for palabra in frecuencia:
        grupos.setdefault(quitar_acentos(palabra), []).append(palabra)
    return {k: sorted(v, key=lambda p: -frecuencia[p]) for k, v in grupos.items() if len(v) > 1}, len(frecuencia)

print(quitar_acentos("canción, año, pequeño, città, perché"))   # dato-es
for idioma in ("spa", "ita"):
    colisiones, n_tipos = tipos_en_colision(resenas[idioma]["texto_limpio"])
    ejemplos = sorted(colisiones.values(), key=lambda v: -len(v))[:6]
    print(idioma, f"→ {len(colisiones)} gruppi di parole che si fondono, su {n_tipos} tipi:", ejemplos)

**Cosa osservi?** Anche in un corpus piccolo ci sono gruppi di parole distinte che restano uguali senza accenti (`que`/`qué`, `se`/`sé`, `llevo`/`llevó` in spagnolo; `e`/`è`, `ne`/`né` in italiano). Nella lezione 6 (sentimento) una negazione o un avverbio di una sola lettera può cambiare la polarità. **Regola pratica:** togli gli accenti solo se i tuoi testi arrivano con accenti incoerenti (come le recensioni rumorose) e il tuo compito lo tollera; nei compiti di analisi grammaticale (lezione 7) non farlo.

### 1.5 Apostrofi, virgolette tipografiche e caratteri invisibili

Copiare e incollare da Word, PDF o dal web porta virgolette arricciate (`’ “ ”`), trattini lunghi e caratteri invisibili (spazio di larghezza zero `U+200B`, trattino morbido `U+00AD`). All'occhio sono innocui; per uno `split()` o un confronto, no: `l’uomo` con apostrofo curvo e `l'uomo` con apostrofo dritto sono stringhe diverse. Li normalizziamo con `str.translate`.

In [ ]:
# datos-es
TEXTO_SUCIO = ["Dijo: “hola” y se fue…", "l’uomo dell’amico", "ca­sa​ y tra​bajo"]

In [ ]:
INVISIBLES = dict.fromkeys(map(ord, "​‌‍⁠﻿­"), None)
TIPOGRAFICAS = {ord("‘"): "'", ord("’"): "'", ord("´"): "'", ord("`"): "'",
                ord("“"): '"', ord("”"): '"'}

def normalizar_unicode(texto, conservar=False):
    """Passo 1: NFC, virgolette e apostrofi dritti, senza caratteri invisibili."""
    texto = unicodedata.normalize("NFC", texto)
    return texto.translate(INVISIBLES).translate(TIPOGRAFICAS)

for t in TEXTO_SUCIO:
    print(repr(t), len(t), "→", repr(normalizar_unicode(t)), len(normalizar_unicode(t)))

**Cosa osservi?** Le lunghezze cambiano e le stringhe restano come quelle che scriveresti a mano. Questo sarà il primo passo di `limpiar`. I puntini di sospensione tipografici «…» restano come sono: non tutto ciò che è strano va cambiato.

---
## 2. Pulizia con espressioni regolari

**Principio guida: pulire non è cancellare.** Ogni cosa che togli la perdi per sempre nel resto della pipeline. Prima di eliminare MAIUSCOLE, `!!!`, emoji o negazioni chiediti quale informazione portano (nella lezione 6 vedrai che il sentimento dipende da esse). Due strategie: **estrarre** quel segnale prima di pulire (un contatore di maiuscole, di esclamativi, di emoji) oppure **sostituirlo** con un token (un'emoji → `<emo_pos>`).

Costruiamo `limpiar(texto)` come una lista di **passi con nome**, ognuno una piccola funzione verificabile. Il primo (`normalizar_unicode`) ce l'hai già. Cominciamo con URL e menzioni.

In [ ]:
RE_URL = re.compile(r"(?:https?://|www\.)\S+")
RE_MENCION = re.compile(r"@\w+")
RE_HASHTAG = re.compile(r"#(\w+)")

def quitar_urls(texto, conservar=False):
    """Passo 2: elimina gli indirizzi web."""
    return RE_URL.sub(" ", texto)

def quitar_menciones_hashtags(texto, conservar=False):
    """Passo 3: toglie le @menzioni; dell'#hashtag toglie il simbolo (conservar=True lascia la parola)."""
    texto = RE_MENCION.sub(" ", texto)
    return RE_HASHTAG.sub(r"\1" if conservar else " ", texto)

MUESTRAS_WEB = ["@atencion_cliente ¡Gracias! Más info en https://t.co/aB12cD #recomendado", "Vedi www.esempio.test/r/4821 #consigliato, @supporto_utenti"]   # dato-es
for t in MUESTRAS_WEB:
    print(repr(quitar_menciones_hashtags(quitar_urls(t))), "|", repr(quitar_menciones_hashtags(quitar_urls(t), conservar=True)))

Ora emoji e residui di markup. Nell'estratto de *I promessi sposi* le parole in corsivo compaiono tra trattini bassi (`_Resegone_`): un residuo dell'edizione elettronica che un tokenizzatore leggerebbe come parte della parola. Un'emoji, invece, è informazione: la sostituiamo con un token in base alla sua polarità *se `conservar=True`* (criterio proprio e discutibile: `🙄` è piuttosto ironica).

> **Attenzione:** i trattini bassi compaiono anche **dentro** i nomi utente (`@atencion_cliente`). Per questo il pattern richiede che il trattino basso di apertura e quello di chiusura non siano attaccati a una lettera all'esterno (`(?<!\w)` e `(?!\w)`). La cella mostra, nella terza colonna, cosa succederebbe con la versione ingenua `_(.+?)_`.

In [ ]:
RE_EMOJI = re.compile("[\U0001F300-\U0001FAFF☀-➿⭐️]+")
EMOJI_POS = set("😊🙂👍😍👌🙌")
EMOJI_NEG = set("👎😡😤🤦🙄")
RE_MARKUP = re.compile(r"(?<!\w)_(\S(?:.*?\S)?)_(?!\w)")

def tratar_emojis(texto, conservar=False):
    """Passo 4: toglie le emoji o le sostituisce con <emo_pos>, <emo_neg> o <emo_neu>."""
    def cambiar(m):
        if not conservar:
            return " "
        g = m.group(0)
        clase = "pos" if any(c in EMOJI_POS for c in g) else "neg" if any(c in EMOJI_NEG for c in g) else "neu"
        return f" <emo_{clase}> "
    return RE_EMOJI.sub(cambiar, texto)

def quitar_markup(texto, conservar=False):
    """Passo 5: toglie i _trattini bassi_ di corsivo che circondano un testo."""
    return RE_MARKUP.sub(r"\1", texto)

MUESTRAS_EMOJI = ["Me encantó 😍👍 pero el envío... 🙄", "Il _Resegone_, così chiamato, e _non lo fanno_.... tutto", "@soporte_ayuda no funciona_"]   # dato-es
for t in MUESTRAS_EMOJI:
    print(repr(tratar_emojis(t, conservar=True)), "|", repr(quitar_markup(t)), "|", repr(re.sub(r"_(.+?)_", r"\1", t)))

Altri passi, con le loro trappole:

- **Allungamenti** (`buenísimooo`, `bellissimooo`): una lettera ripetuta **tre o più volte** si riduce a una (in spagnolo e italiano nessuna parola ha tre lettere uguali di fila, ma sì *due*: `llegó`, `bello`). Limitiamo la regola alle **lettere**: con `\w` un `1000` diventerebbe `10`.
- **Punteggiatura ripetuta** (`!!!`, `???`): si riduce a un segno. L'intensità si perde, per questo si estrae prima.
- **Spazi**: comprimere i multipli e attaccare la punteggiatura alla parola precedente (`idea , ejecución` → `idea, ejecución`).

In [ ]:
RE_ALARGADA = re.compile(r"([^\W\d_])\1{2,}")

def reducir_alargamientos(texto, conservar=False):
    """Passo 6: riduce a una le lettere ripetute tre o più volte (solo lettere, mai cifre)."""
    return RE_ALARGADA.sub(r"\1", texto)

def reducir_puntuacion(texto, conservar=False):
    """Passo 7: riduce !!! e ??? a un solo segno e i puntini di sospensione lunghi a tre."""
    texto = re.sub(r"([!?¡¿])\1+", r"\1", texto)      # dato-es
    return re.sub(r"\.{4,}", "...", texto)

def arreglar_espacios(texto, conservar=False):
    """Passo 8: comprime gli spazi e attacca la punteggiatura alla parola precedente."""
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(r"\s+([,;:.!?])", r"\1", texto)
    return texto.strip()

MUESTRAS_RUIDO = ["buenísimooo!!!  llegó tardeeee ...., pero 1000 gracias", "bellissimooo!!  davvero ???", "Buena idea , ejecución mejorable"]   # dato-es
for t in MUESTRAS_RUIDO:
    print(repr(arreglar_espacios(reducir_puntuacion(reducir_alargamientos(t)))))
print(repr(re.sub(r"(\w)\1{2,}", r"\1", "cuesta 1000 euros")), "← la versione con \\w rompe le cifre")

### 2.1 Assemblare `limpiar` e vedere l'effetto di ogni passo

Riuniamo gli otto passi in una lista con nome. `mostrar_pasos` stampa il testo dopo ogni passo che lo modifica.

In [ ]:
PASOS_LIMPIEZA = [
    ("unicode", normalizar_unicode),
    ("URLs", quitar_urls),
    ("menzioni e hashtag", quitar_menciones_hashtags),
    ("emojis", tratar_emojis),
    ("markup", quitar_markup),
    ("allungamenti", reducir_alargamientos),
    ("punteggiatura ripetuta", reducir_puntuacion),
    ("spazi", arreglar_espacios),
]

def limpiar(texto, conservar=False):
    """Pulisce un testo applicando i passi in ordine. conservar=True lascia hashtag ed emoji come segnale."""
    for nombre, paso in PASOS_LIMPIEZA:
        texto = paso(texto, conservar)
    return texto

def mostrar_pasos(texto, conservar=False):
    print("originale :", repr(texto))
    for nombre, paso in PASOS_LIMPIEZA:
        nuevo = paso(texto, conservar)
        if nuevo != texto:
            print(f"{nombre:>22} :", repr(nuevo))
        texto = nuevo

In [ ]:
# datos-es
RUIDOSAS = {"spa": "@atencion_cliente ¡¡BUENÍSIMOOOO!!!   El pedido llegó en 2 días por 4,99 € 😍😍 https://t.co/aB12cD #recomendado",
            "ita": "@assistenza Spedizione velocissimaaa!!!  Pagato 4,99 € e arrivato in 2 giorni 😍 https://t.co/aB12cD #consigliato"}

In [ ]:
mostrar_pasos(RUIDOSAS["spa"])
print()
mostrar_pasos(RUIDOSAS["ita"], conservar=True)

**Cosa osservi?** Ogni passo fa una cosa e si può provare da solo; se una regex è troppo avida, il risultato intermedio rivela **quale**. Nota che `4,99` sopravvive intatto (non abbiamo toccato le cifre) e che con `conservar=True` l'emoji diventa `<emo_pos>` e l'hashtag lascia la sua parola.

Una trappola classica: togliere «tutta la punteggiatura» con `re.sub(r"[^\w\s]", "", texto)` trasformerebbe `4,99` in `499`. Verificalo qui sotto e guarda anche quali segnali espressivi si perdono pulendo e come estrarli **prima**.

In [ ]:
NEGADORES = {"spa": {"no", "nunca", "ni", "tampoco", "sin"}, "ita": {"non", "mai", "né", "senza", "nessun", "nessuno"}}   # dato-es

def senales(texto, idioma):
    """Conta i segnali espressivi PRIMA di pulire."""
    palabras = re.findall(r"\w+", texto)
    return {"maiuscole": sum(1 for p in palabras if len(p) >= 3 and p.isupper()),
            "esclamativi": texto.count("!"),
            "emojis": len(RE_EMOJI.findall(texto)),
            "allungate": len(RE_ALARGADA.findall(texto)),
            "negazioni": sum(1 for p in palabras if p.lower() in NEGADORES[idioma])}

print(re.sub(r"[^\w\s]", "", "cuesta 4,99 euros"), "← il prezzo si è rotto")   # dato-es
for idioma in ("spa", "ita"):
    print(idioma, senales(RUIDOSAS[idioma], idioma))
    print("   dopo la pulizia:", senales(limpiar(RUIDOSAS[idioma]), idioma))

**Cosa osservi?** Dopo `limpiar`, le emoji e le parole allungate scendono a zero e gli esclamativi passano da 3 a 1: **quell'informazione non è più nel testo**. Le MAIUSCOLE restano lì perché `limpiar` non passa al minuscolo; si perdono quando lo facciamo (come in `palabras`). Se il compito lo richiede (sentimento), la si calcola prima e la si aggiunge come feature a parte, oppure la si conserva con un token. Le negazioni, invece, **non si toccano** nella pulizia: si decide cosa farne nelle lezioni 3 (stopword) e 6.

---
## 3. Tokenizzazione

Un **token** è l'unità con cui lavora il resto della pipeline. Qui tokenizziamo in **parole** (più la punteggiatura). Basta `split()`? Proviamo con frasi che contengono le trappole dello spagnolo e dell'italiano: punteggiatura attaccata, apostrofi con elisione (`l'uomo`, `un'ora`, `dell'amico`), contrazioni (`del`, `al`), segni di apertura `¿ ¡`, cifre con la virgola e abbreviazioni.

In [ ]:
# datos-es
FRASES_TOKEN = {
    "spa": "¿Cuánto cuesta el pan del Sr. García? ¡Solo 4,99 euros al día!",
    "ita": "L'uomo dell'amico ha aspettato un'ora, all'una e mezza: costava 4,99 euro.",
}

In [ ]:
for idioma, frase in FRASES_TOKEN.items():
    print(idioma, "split():", frase.split())

**Cosa osservi?** `García?` e `día!` hanno il segno attaccato; `¿Cuánto` e `¡Solo` si trascinano quello di apertura; `un'ora,` unisce articolo, nome e virgola. Un sacco di parole vedrebbe `día!` e `día` come parole diverse.

### 3.1 Un tokenizzatore con `re.findall`

Un pattern con più alternative, dalla più specifica alla più generale: (1) token della forma `<...>` (quelli che creiamo pulendo), (2) **numeri** con separatori (`4,99`, `1.250,50`), (3) **elisione**: parola + apostrofo davanti a un'altra lettera (`l'`, `dell'`, `un'`), (4) parole (con trattini interni), (5) qualunque altro simbolo isolato (punteggiatura, `¿`, `¡`).

In [ ]:
PATRON_TOKEN = re.compile(r"<\w+>|\d+(?:[.,]\d+)*|\w+'(?=\w)|\w+(?:-\w+)*|[^\w\s]")

def tokenizar(texto):
    """Tokenizza in parole, numeri, elisioni e segni di punteggiatura."""
    return PATRON_TOKEN.findall(texto)

for idioma, frase in FRASES_TOKEN.items():
    print(idioma, tokenizar(frase))

### 3.2 NLTK e spaCy a confronto

Confrontiamo il nostro tokenizzatore con `nltk.tokenize.word_tokenize(language=...)` e con il tokenizzatore di spaCy (`spacy.blank`, senza modello: solo regole). **Non esiste una tokenizzazione «corretta» universale**: ogni strumento prende decisioni diverse ed è bene conoscerle.

In [ ]:
codigo_spacy = {"spa": "es", "ita": "it"}   # dato-es
nlp_blank = {idioma: spacy.blank(codigo) for idioma, codigo in codigo_spacy.items()}
nombre_nltk = {"spa": "spanish", "ita": "italian"}

def comparar_tokenizadores(texto, idioma):
    return {"split": texto.split(), "re.findall": tokenizar(texto),
            "nltk": word_tokenize(texto, language=nombre_nltk[idioma]),
            "spaCy": [t.text for t in nlp_blank[idioma](texto)]}

for idioma, frase in FRASES_TOKEN.items():
    print(idioma, frase)
    for herramienta, tokens in comparar_tokenizadores(frase, idioma).items():
        print(f"   {herramienta:>10} ({len(tokens):2}):", tokens)

**Cosa osservi?** (1) NLTK lascia `¿Cuánto` e `¡Solo` attaccati e **non separa** gli apostrofi dell'italiano (`l'uomo` resta intero). (2) spaCy separa `¿` e `¡` e taglia `L'` / `uomo`, come fa il nostro pattern. (3) Nessuno spezza `del` né `al`: in spagnolo li trattiamo come parole (più avanti, con `es_core_news_sm`, vedrai che spaCy le scompone in `de + el`, un passo di analisi e non di tokenizzazione pura). (4) Le cifre `4,99` si conservano intere. (5) `Sr.` è un caso limite: con il nostro pattern resta `Sr` + `.`. Cifre di spaCy: con spaCy 3.8 e `spacy.blank`; possono variare leggermente con altre versioni.

### 3.3 Segmentazione in frasi

Separare le frasi sembra facile («tagliare a ogni punto») e non lo è: i punti compaiono in abbreviazioni (`Sr.`, `Dott.`, `p.m.`), sigle (`U.S.A.`) e decimali (`4.5`). `sent_tokenize` di NLTK usa un modello addestrato (Punkt) che azzecca molto, ma non sempre.

In [ ]:
# datos-es
TEXTOS_FRASES = {
    "spa": "El Sr. García pagó 4.5 euros. Luego se fue, etc. Volvió a las 3 p.m. y dijo: ¿Qué tal? Bien. Vi a la U.S.A. con el Dr. Pérez.",
    "ita": "Il dott. Rossi ha pagato 4.5 euro. Poi è andato via, ecc. Tornò alle 3 e disse: Bene. Sig.ra Bianchi, buongiorno.",
}

In [ ]:
for idioma, texto in TEXTOS_FRASES.items():
    ingenua = [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto)]
    punkt = sent_tokenize(texto, language=nombre_nltk[idioma])
    print(idioma, "regex ingenua :", len(ingenua), ingenua)
    print(idioma, "sent_tokenize :", len(punkt), punkt)

**Cosa osservi?** Il testo in spagnolo ha 5 frasi e quello in italiano 4. La regex ingenua spezza a ogni abbreviazione (9 e 5 «frasi»). Punkt sbaglia meno (7 e 4), ma fallisce anch'esso: taglia dopo `p.m.` e `U.S.A.` in spagnolo e, in italiano, taglia dopo `Dott.` e **unisce** due frasi dopo `ecc.`; azzecca il numero, ma non i tagli (con la versione di NLTK usata qui; può variare tra versioni). **Regola pratica:** se la segmentazione in frasi conta (riassunti, lezione 11), controlla a mano un campione dei tuoi testi; se non conta, non farla.

---
## 4. Token e tipi

Un **token** è ogni occorrenza di una parola nel testo; un **tipo** (*type*) è una parola distinta del vocabolario. `il gatto e il cane` ha 5 token e 4 tipi. Con essi si calcolano la **frequenza** di ogni tipo, il **rapporto tipi/token** (TTR) e gli ***hapax legomena***, i tipi che compaiono **una sola volta**. Vedremo che pochi tipi concentrano quasi tutti i token e che la maggior parte dei tipi sono hapax.

Calcoliamo le cifre per i due libri (puliti con `limpiar` e passati al minuscolo) e per le recensioni. **Avvertenza:** sono testi di secoli e stili diversi (lo spagnolo del Seicento di Cervantes, l'italiano dell'Ottocento di Manzoni e recensioni sintetiche di stile attuale), quindi le differenze non dicono nulla su «lo spagnolo» né «l'italiano».

In [ ]:
def palabras(texto):
    """Token di parola in minuscolo (senza punteggiatura)."""
    return [t for t in tokenizar(texto.lower()) if re.match(r"\w", t)]

def resumen_tokens(tokens):
    conteo = Counter(tokens)
    hapax = sum(1 for v in conteo.values() if v == 1)
    return {"tokens": len(tokens), "tipi": len(conteo), "TTR": round(len(conteo) / len(tokens), 3),
            "hapax": hapax, "% hapax sui tipi": round(100 * hapax / len(conteo), 1)}

corpus = {}
for idioma in ("spa", "ita"):
    corpus[f"libro {idioma}"] = palabras(limpiar(libros[idioma]))
    corpus[f"recensioni {idioma}"] = [p for t in resenas[idioma]["texto"] for p in palabras(limpiar(t))]

tabla = pd.DataFrame([resumen_tokens(tk) for tk in corpus.values()], index=list(corpus))
print(tabla.to_string())

**Cosa osservi?** Il TTR **non è confrontabile** tra corpus di dimensioni diverse: più token ci sono, più i tipi si ripetono e più scende il rapporto. Per confrontare bisogna fissare lo stesso numero di token (per esempio i primi 1 000). Lo facciamo qui sotto e aggiungiamo le 20 parole più frequenti.

In [ ]:
primeros = pd.DataFrame([resumen_tokens(tk[:1000]) for tk in corpus.values()], index=list(corpus))
print("Primi 1 000 token di ogni corpus:")
print(primeros.to_string())
print()
top = {nombre: Counter(tk).most_common(20) for nombre, tk in corpus.items()}
for nombre, lista in top.items():
    print(f"{nombre}:", ", ".join(f"{p} {n}" for p, n in lista))

### 4.1 La distribuzione rango-frequenza

Ordiniamo i tipi dal più al meno frequente e disegniamo frequenza contro rango in scala doppio-logaritmica, insieme alla curva di crescita del vocabolario (quanti tipi ci sono dopo aver letto *n* token).

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for nombre in corpus:
    tokens = corpus[nombre]
    frecuencias = sorted(Counter(tokens).values(), reverse=True)
    ejes[0].loglog(range(1, len(frecuencias) + 1), frecuencias, label=nombre)
    crecimiento = [len(set(tokens[:n])) for n in range(100, len(tokens) + 1, 100)]
    ejes[1].plot(range(100, 100 * len(crecimiento) + 1, 100), crecimiento, label=nombre)
ejes[0].set_xlabel("rango (scala log)")
ejes[0].set_ylabel("frequenza (scala log)")
ejes[0].set_title("Rango-frequenza")
ejes[1].set_xlabel("token letti")
ejes[1].set_ylabel("tipi distinti")
ejes[1].set_title("Crescita del vocabolario")
ejes[0].legend()
plt.tight_layout()
plt.show()

**Cosa osservi?** A sinistra, quasi una retta discendente in scala log-log: pochi tipi (`de`, `y`, `que`, `e`, `di`, `il`…) accumulano moltissimi token e una lunga coda di tipi compare una volta (la tendenza è nota come legge di Zipf). A destra, la curva del vocabolario **continua a crescere** senza stabilizzarsi: compariranno sempre parole nuove, cosa che nelle lezioni 4 e 8 si traduce nel problema delle parole fuori vocabolario (OOV). Più della metà dei tipi sono hapax in tutti e quattro i corpora.

---
## 5. Misurare la qualità di una pulizia

Le recensioni hanno la colonna `texto_limpio`: una versione di riferimento (gold) scritta a mano. Con essa possiamo **misurare** una pulizia invece di giudicarla a occhio. Tre metriche semplici, confrontando `limpiar(texto)` con `texto_limpio` **dopo la normalizzazione** (NFC, `casefold`, spazi), perché `limpiar` non ripristina le maiuscole:

1. **Corrispondenza esatta**: percentuale di recensioni il cui risultato è identico al gold.
2. **F1 sui token**: precisione e richiamo (*recall*) sul multiinsieme delle parole (media per recensione).
3. **Somiglianza** di `difflib.SequenceMatcher`: rapporto tra 0 e 1 a livello di carattere.

In [ ]:
def normalizar_comparacion(texto):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", texto).casefold()).strip()

def f1_tokens(pred, gold):
    """F1 tra due liste di token (multiinsiemi)."""
    comunes = sum((Counter(pred) & Counter(gold)).values())
    if comunes == 0:
        return 0.0
    precision, recall = comunes / len(pred), comunes / len(gold)
    return 2 * precision * recall / (precision + recall)

def evaluar_limpieza(limpiador, textos, golds):
    pares = [(normalizar_comparacion(limpiador(t)), normalizar_comparacion(g)) for t, g in zip(textos, golds)]
    return {"esatta": round(np.mean([p == g for p, g in pares]), 3),
            "F1": round(np.mean([f1_tokens(palabras(p), palabras(g)) for p, g in pares]), 3),
            "somiglianza": round(np.mean([difflib.SequenceMatcher(None, p, g).ratio() for p, g in pares]), 3)}

def solo_espacios(texto):
    return arreglar_espacios(texto)

resultados = {}
for idioma in ("spa", "ita"):
    d = resenas[idioma]
    resultados[f"senza pulire {idioma}"] = evaluar_limpieza(lambda t: t, d["texto"], d["texto_limpio"])
    resultados[f"solo spazi {idioma}"] = evaluar_limpieza(solo_espacios, d["texto"], d["texto_limpio"])
    resultados[f"limpiar {idioma}"] = evaluar_limpieza(limpiar, d["texto"], d["texto_limpio"])
    resultados[f"limpiar(conservar) {idioma}"] = evaluar_limpieza(lambda t: limpiar(t, conservar=True), d["texto"], d["texto_limpio"])
print(pd.DataFrame(resultados).T.to_string())

**Cosa osservi?** (1) `limpiar` migliora tutte e tre le metriche rispetto al non fare nulla. (2) La versione `conservar=True` **ottiene un punteggio peggiore**: non è un errore, è che il gold **cancella** hashtag ed emoji mentre noi li conserviamo di proposito. Una metrica misura la distanza da *un* criterio; se il criterio non è il tuo, la cifra inganna. (3) Anche così l'esatta non arriva al 100 %: resta ciò che nessuna regola può recuperare. Lo classifichiamo.

In [ ]:
def sin_puntuacion(texto):
    return re.sub(r"[^\w\s]", " ", texto)

def causa_residuo(pred, gold):
    """Classifica perché una pulizia non coincide con il gold."""
    p, g = normalizar_comparacion(pred), normalizar_comparacion(gold)
    if p == g:
        return "coincide"
    if palabras(p) == palabras(g):
        return "solo punteggiatura"
    if palabras(quitar_acentos(p)) == palabras(quitar_acentos(g)):
        return "accenti persi"
    return "altre (refusi, abbreviazioni)"

for idioma in ("spa", "ita"):
    d = resenas[idioma]
    causas = Counter(causa_residuo(limpiar(t), g) for t, g in zip(d["texto"], d["texto_limpio"]))
    print(idioma, {k: f"{100 * v / len(d):.1f}%" for k, v in causas.most_common()})

**Cosa osservi?** Una parte delle recensioni continua a non coincidere per la **punteggiatura** (il gold cambia `!!` in `.`: è una convenzione del gold, non un difetto), un'altra per **accenti persi** (`rompio` invece di `rompió`) e un'altra per **refusi e abbreviazioni** (`esrellas`, `q`): la regex non può sapere che `esrellas` significa `estrellas`. Recuperare gli accenti e correggere i refusi richiede un dizionario o un modello (un correttore ortografico), e anche così sbaglierà a volte. **L'importante è sapere cosa può e cosa non può fare la pulizia e misurarlo.**

In [ ]:
# Esempi concreti di ciò che la pulizia non recupera (spagnolo e italiano)
for idioma in ("spa", "ita"):
    d = resenas[idioma]
    vistos = 0
    for t, g in zip(d["texto"], d["texto_limpio"]):
        if causa_residuo(limpiar(t), g) != "coincide" and vistos < 3 and len(t) < 110:
            print(idioma, "|", limpiar(t))
            print("   gold |", g)
            vistos += 1

---
## 6. Altre difficoltà del linguaggio (che la pulizia non risolve)

Nella lezione 1 abbiamo visto l'ambiguità lessicale e sintattica. Il testo grezzo aggiunge tre altre fonti di difficoltà: l'**anafora** (a chi si riferisce un pronome), il **contesto e la pragmatica** (l'implicito e l'ironia) e la **variazione** (refusi, registri, emoji, mescolanza di lingue). Le prime due le mostriamo soltanto: sono limiti dell'approccio classico. La terza la possiamo **quantificare** sulle recensioni.

### 6.1 Ambiguità referenziale (anafora)

In [ ]:
# datos-es
REFERENCIAL = [
    ("Ana le dijo a Marta que ella tenía razón", "Ana o Marta?"),
    ("El trofeo no cabe en la maleta porque es demasiado grande", "el trofeo"),
    ("El trofeo no cabe en la maleta porque es demasiado pequeña", "la maleta"),
    ("Il trofeo non entra nella valigia perché è troppo grande", "il trofeo"),
    ("Marco ha detto a Luca che lui aveva ragione", "Marco o Luca?"),
]

In [ ]:
for frase, referente in REFERENCIAL:
    print(f"{frase}  →  {referente}")

**Cosa osservi?** Cambiando una sola parola (`grande` / `pequeña`) cambia a cosa si riferisce `es`; e nella prima e nell'ultima frase non c'è nemmeno una risposta unica. Risolverlo richiede **conoscenza del mondo**, non solo grammatica: quasi nessun metodo classico lo fa bene, ed è un limite da conoscere. Nessuna regex e nessuna pulizia lo risolve.

### 6.2 Contesto e pragmatica: l'ironia nelle recensioni

Ciò che si dice non è sempre ciò che si vuole dire: `¡Qué buen día para que se estropee el aire acondicionado!` è un lamento; `Bravo, hai proprio scelto il momento giusto!` è un rimprovero. Lo verifichiamo con le recensioni difficili (`dificil == 1`) etichettate come **negative** che contengono parole di tono positivo: per una regola a parole sembrerebbero positive.

In [ ]:
# datos-es
TONO_POSITIVO = {"spa": {"maravilla", "fantástico", "genial", "perfecto", "gracias", "felicidades", "excelente"},
                 "ita": {"bravo", "complimenti", "grazie", "ottimo", "perfetto", "bellissima", "bellissimo", "eccellente"}}

In [ ]:
for idioma in ("spa", "ita"):
    d = resenas[idioma]
    dificiles = d[(d["dificil"] == 1) & (d["etiqueta"] == "negativo")]
    con_tono = dificiles[[bool(set(palabras(limpiar(t))) & TONO_POSITIVO[idioma]) for t in dificiles["texto"]]]
    print(f"{idioma}: {len(dificiles)} recensioni difficili negative; con parole di tono positivo: {len(con_tono)}")
    for t in con_tono["texto"].head(3):
        print("   ·", limpiar(t))

**Cosa osservi?** Sono lamentele scritte con parole elogiative. Una pulizia impeccabile non le trasforma in altro: il problema non è nel testo grezzo, è nella **pragmatica**. Per questo l'analisi del sentimento ha un tetto (lezione 6) e bisogna analizzare gli errori a mano; e per questo, nella lezione 12, la si confronta con un modello neurale che vede più contesto.

### 6.3 Variazione: quantificarla nelle recensioni

La variazione si può invece **misurare**: con semplici contatori calcoliamo la percentuale di recensioni che presentano ciascun fenomeno. Per gli accenti persi usiamo il gold (possibile solo perché lo abbiamo).

In [ ]:
# datos-es
CHAT = {"spa": {"q", "xq", "pq", "xk", "k", "tb", "tbn", "ke"}, "ita": {"ke", "xke", "nn", "sn", "cmq", "x", "qlc", "tvb"}}
TILDES = set("áéíóúàèìòùñÁÉÍÓÚÀÈÌÒÙ")

In [ ]:
def pct(condicion, textos):
    return round(100 * np.mean([bool(condicion(t)) for t in textos]), 1)

def variacion(idioma):
    d = resenas[idioma]
    textos, golds = d["texto"], d["texto_limpio"]
    n_tildes = lambda t: sum(c in TILDES for c in t)
    return {"qualche parola in MAIUSCOLO": pct(lambda t: senales(t, idioma)["maiuscole"] > 0, textos),
            "accenti persi": round(100 * np.mean([n_tildes(t) < n_tildes(g) for t, g in zip(textos, golds)]), 1),
            "allungamenti": pct(RE_ALARGADA.search, textos),
            "emojis": pct(RE_EMOJI.search, textos),
            "URLs": pct(RE_URL.search, textos),
            "menzioni o hashtag": pct(lambda t: re.search(r"[@#]\w", t), textos),
            "spazi doppi": pct(lambda t: "  " in t, textos),
            "abbreviazioni da chat": pct(lambda t: set(re.findall(r"\w+", t.lower())) & CHAT[idioma], textos),
            "testo diverso dal gold": round(100 * np.mean([t != g for t, g in zip(textos, golds)]), 1)}

tabla_variacion = pd.DataFrame({"spa": variacion("spa"), "ita": variacion("ita")})
print(tabla_variacion.to_string())
tabla_variacion.plot.barh(figsize=(9, 4.5))
plt.xlabel("% di recensioni")
plt.title("Variazione nelle recensioni grezze (300 per lingua)")
plt.tight_layout()
plt.show()

**Cosa osservi?** Quasi la metà delle recensioni differisce dal gold per qualche causa, e ogni fenomeno colpisce una frazione piccola ma non trascurabile. Ogni variante è, per un programma che confronta stringhe, una parola diversa: `holaaa` e `hola`, `xq` e `porque`. La normalizzazione (questa lezione) e la lemmatizzazione (lezione 3) riducono una parte del problema, ma non tutto. E sono cifre di recensioni **sintetiche** con rumore iniettato: nei testi reali le proporzioni saranno altre.

---
## 7. Riepilogo e passo successivo

- **Unicode**: un carattere non è un byte; NFC e NFD danno stringhe diverse per ciò che vedi uguale; normalizza in NFC alla lettura e usa `casefold()` per confrontare.
- **Accenti**: toglierli riduce le varianti ma cancella distinzioni (`papá`/`papa`, `è`/`e`); fallo solo con criterio e proteggi la `ñ`.
- **Pulizia**: passi con nome, ognuno verificabile da solo; **pulire non è cancellare**: estrai o sostituisci con token ciò che porta informazione (emoji, maiuscole, `!!!`).
- **Tokenizzazione**: `split()` non basta; un pattern con `re.findall` gestisce apostrofi, `¿ ¡`, cifre; NLTK e spaCy prendono decisioni diverse.
- **Token e tipi**: pochi tipi concentrano molti token e più della metà dei tipi sono hapax; il TTR dipende dalla dimensione.
- **Valutare** una pulizia rispetto a un gold con esatta, F1 e somiglianza, e sapere cosa non può recuperare (accenti, refusi).
- L'**anafora** e la **pragmatica** non si risolvono pulendo: sono limiti che rivedrai nelle lezioni 6 e 12.

**Prossimo passo:** nella lezione 3 toglierai le *stopword* con criterio e confronterai *stemming* e lemmatizzazione. Prima, fai il notebook di pratica di questa lezione.